## Import Libraries

In [4]:
#!pip install pycox torchtuples h5py
#!pip install lifelines
#!pip install --upgrade scikit-survival

In [5]:
import h5py
import torch
import numpy as np
import pandas as pd
import os
import torch.nn as nn
from pycox.models import CoxPH
import torchtuples as tt
from pycox.evaluation import EvalSurv
from lifelines import CoxPHFitter
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Import Original Data

In [10]:
np.random.seed(48764388)
_ = torch.manual_seed(48764388)

In [11]:
# =====================================================================
# Load the author's .h5 data file (Upload the file to Google Colab)
# - metabric_IHC4_clinical_train_test.h5
# =====================================================================

file_path = 'metabric_IHC4_clinical_train_test.h5'

# Check if the file exists
if os.path.exists(file_path):
    with h5py.File(file_path, 'r') as f:
        # Extract Train / Test datasets
        x_train = f['train']['x'][:].astype('float32')
        e_train = f['train']['e'][:].astype('int32')
        t_train = f['train']['t'][:].astype('float32')

        x_test = f['test']['x'][:].astype('float32')
        e_test = f['test']['e'][:].astype('int32')
        t_test = f['test']['t'][:].astype('float32')

    print("✅ .h5 data file successfully loaded!")
    print(f"- Number of train patients: {x_train.shape[0]} / Number of input features: {x_train.shape[1]}")
    print(f"- Number of test patients: {x_test.shape[0]}")
else:
    print(f"⚠️ File '{file_path}' not found. Please verify the file path.")

✅ .h5 data file successfully loaded!
- Number of train patients: 1523 / Number of input features: 9
- Number of test patients: 381


In [12]:
file_path = 'metabric_IHC4_clinical_train_test.h5'

# Extract data
with h5py.File(file_path, 'r') as f:
    x_train = f['train']['x'][:]
    e_train = f['train']['e'][:]
    t_train = f['train']['t'][:]

# Feature names matching Table 1 and dataset description
column_names = [
    'MKI67',
    'EGFR',
    'PGR',
    'ERBB2',
    'Hormone',
    'Radio',
    'Chemo',
    'ER-pos',
    'Age'
]

# Create a pandas DataFrame
df_train = pd.DataFrame(x_train, columns=column_names)
df_train['Event (e)'] = e_train
df_train['Time (t)'] = t_train

# Inspect the top 5 patient records
print("📊 Top 5 samples of Train data:")
df_train.head()

📊 Top 5 samples of Train data:


,MKI67,EGFR,PGR,ERBB2,Hormone,Radio,Chemo,ER-pos,Age,Event (e),Time (t)
0,5.603834,7.811392,10.797988,5.967607,1.0,1.0,0.0,1.0,56.840000,0,99.333336
1,5.284882,9.581043,10.204620,5.664970,1.0,0.0,0.0,1.0,85.940002,1,95.733330
2,5.920251,6.776564,12.431715,5.873857,0.0,1.0,0.0,1.0,48.439999,0,140.233337
3,6.654017,5.341846,8.646379,5.655888,0.0,0.0,0.0,0.0,66.910004,0,239.300003
4,5.456747,5.339741,10.555724,6.008429,1.0,0.0,0.0,1.0,67.849998,1,56.933334


In [13]:
# Format target variables as (duration, event) tuples
y_train = (t_train, e_train)
y_test = (t_test, e_test)

# Split validation set (20% of the training data)
val_size = int(len(x_train) * 0.2)
x_val = x_train[:val_size]
y_val = (t_train[:val_size], e_train[:val_size])

x_train_sub = x_train[val_size:]
y_train_sub = (t_train[val_size:], e_train[val_size:])

## Model Development (DeepSurv)

In [14]:
# =====================================================================
# Build DeepSurv Neural Network Architecture (Tuned)
# =====================================================================
in_features = x_train.shape[1]   # Number of input features (9)
num_nodes = [32, 32]             # Hidden layers
out_features = 1                 # Risk score output
dropout = 0.1

# Apply SELU activation function to enhance neural network performance
net = tt.practical.MLPVanilla(
    in_features,
    num_nodes,
    out_features,
    batch_norm=True,
    dropout=dropout,
    activation=nn.SELU # Apply SELU activation function
)

# Adjust learning rate (0.01 -> 0.001) for finer optimization
optimizer = tt.optim.Adam(lr=0.001, weight_decay=1e-4)
model = CoxPH(net, optimizer)

print("✅ Model initialized with tuned hyperparameters!")

✅ Model initialized with tuned hyperparameters!


In [15]:
# =====================================================================
# DeepSurv Model Training
# =====================================================================
batch_size = 256
epochs = 100
callbacks = [tt.callbacks.EarlyStopping(patience=10)]

print("\n🚀 Starting DeepSurv model training...")
log = model.fit(
    x_train_sub,
    y_train_sub,
    batch_size=batch_size,
    epochs=epochs,
    callbacks=callbacks,
    val_data=(x_val, y_val),
    verbose=True
)


🚀 Starting DeepSurv model training...
0:	[0s / 0s],		train_loss: 4.8128,	val_loss: 4.9502
1:	[0s / 0s],		train_loss: 4.7755,	val_loss: 4.9512
2:	[0s / 0s],		train_loss: 4.7490,	val_loss: 4.9428
3:	[0s / 0s],		train_loss: 4.7383,	val_loss: 4.9257
4:	[0s / 0s],		train_loss: 4.7211,	val_loss: 4.9146
5:	[0s / 0s],		train_loss: 4.7060,	val_loss: 4.9127
6:	[0s / 0s],		train_loss: 4.7074,	val_loss: 4.9113
7:	[0s / 0s],		train_loss: 4.7206,	val_loss: 4.9090
8:	[0s / 0s],		train_loss: 4.6804,	val_loss: 4.9064
9:	[0s / 0s],		train_loss: 4.6933,	val_loss: 4.9026
10:	[0s / 0s],		train_loss: 4.6802,	val_loss: 4.9002
11:	[0s / 0s],		train_loss: 4.7052,	val_loss: 4.8969
12:	[0s / 0s],		train_loss: 4.6656,	val_loss: 4.8953
13:	[0s / 0s],		train_loss: 4.6714,	val_loss: 4.8966
14:	[0s / 0s],		train_loss: 4.6778,	val_loss: 4.8976
15:	[0s / 0s],		train_loss: 4.6709,	val_loss: 4.8988
16:	[0s / 0s],		train_loss: 4.6717,	val_loss: 4.9029
17:	[0s / 0s],		train_loss: 4.6735,	val_loss: 4.8994
18:	[0s / 0s],		t

In [17]:
# =====================================================================
# [Key Result] Evaluate and Display Test C-index (Concordance Index)
# =====================================================================
# Predict survival probabilities/hazards on the test dataset
_ = model.compute_baseline_hazards()
surv = model.predict_surv_df(x_test)

# Calculate C-index using the survival evaluation class (EvalSurv)
ev = EvalSurv(surv, t_test, e_test, censor_surv='km')
c_index = ev.concordance_td()

print("\n" + "="*50)
print(f"📊 [Replication Result] METABRIC Test C-index: {c_index:.4f}")
print("="*50)


📊 [Replication Result] METABRIC Test C-index: 0.6260


## Model Development (CPH)

In [18]:
# Convert to pandas DataFrames (Train & Test)
df_train = pd.DataFrame(x_train)
df_train['t'] = t_train
df_train['e'] = e_train

df_test = pd.DataFrame(x_test)
df_test['t'] = t_test
df_test['e'] = e_test

# Fit and evaluate the Cox Proportional Hazards (CPH) model
cph = CoxPHFitter()
cph.fit(df_train, duration_col='t', event_col='e')

# Measure the Test C-index
cph_c_index = cph.score(df_test, scoring_method="concordance_index")
print(f"📊 CPH Test C-index: {cph_c_index:.4f}")

📊 CPH Test C-index: 0.6324


## Model Development (RSF)

In [19]:
y_train_struct = np.array(
    list(zip(e_train.astype(bool), t_train)),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)

y_test_struct = np.array(
    list(zip(e_test.astype(bool), t_test)),
    dtype=[('Status', '?'), ('Survival_in_days', '<f8')]
)

print("✅ Structured array conversion for RSF completed!")

# -------------------------------------------------------------------------
# 3. Initialize and Train Random Survival Forest (RSF) Model
# -------------------------------------------------------------------------
rsf = RandomSurvivalForest(
    n_estimators=100,        # Number of trees (default: 100)
    min_samples_split=10,    # Minimum samples required to split a node
    min_samples_leaf=15,     # Minimum samples required at a leaf node
    max_features="sqrt",     # Number of features to consider when looking for the best split
    n_jobs=-1,               # Parallel processing (speed optimization)
    random_state=42
)

print("⏳ Starting RSF model training...")
rsf.fit(x_train, y_train_struct)
print("✅ RSF model training completed!")

# -------------------------------------------------------------------------
# 4. Model Evaluation (Compute C-index / Concordance Index)
# -------------------------------------------------------------------------
# Train C-index
c_index_train = rsf.score(x_train, y_train_struct)

# Test C-index
c_index_test = rsf.score(x_test, y_test_struct)

print("\n" + "="*40)
print(f"📊 RSF Performance on METABRIC Dataset:")
print(f"- Train C-index: {c_index_train:.4f}")
print(f"- Test C-index : {c_index_test:.4f}")
print("="*40)

✅ Structured array conversion for RSF completed!
⏳ Starting RSF model training...
✅ RSF model training completed!

📊 RSF Performance on METABRIC Dataset:
- Train C-index: 0.7394
- Test C-index : 0.6443
